In [10]:
#[CELL 1] STUDENT CONFIGURATION
import csv, json, shutil
from collections import Counter
from pathlib import Path

LAST_NAME = "Silverio"
STUDENT_ID = "TUPM-26-1261"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())

TARGET_YEAR = 2015 + SEED_DIGIT
LOSS_FIELD = "Financial Loss (in Million $)"
COUNT_FIELD = "Attack Type"

print("Seed:", SEED_DIGIT)
print("Target year:", TARGET_YEAR)

Seed: 1
Target year: 2016


In [11]:
#[CELL 2] WORKSPACE AND FILE PATHS
BASE = Path(".")
DATA_DIR = BASE / "data"
OUTPUT_DIR = BASE / "output"
BACKUP_DIR = BASE / "backup"

for folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    folder.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex3_filtered.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex3_top5.json"
report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex3_report.txt"
backup_path = BACKUP_DIR / json_path.name

print("Workspace:", BASE.resolve())
print("Dataset exists:", dataset_path.exists())

Workspace: C:\Users\anica\Lab6_File_Handling_Silverio\Lab6_File_Handling_Additional_Exercises
Dataset exists: True


In [12]:
#[CELL 3] READ DATASET AND SELECT TARGET-YEAR INCIDENTS
total_records = 0
invalid_loss = 0
attack_counts = Counter()
selected = []          
total_loss = 0.0

with open(dataset_path, "r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)
    reader.fieldnames = [name.strip() for name in reader.fieldnames]
    fieldnames = reader.fieldnames

    for order, row in enumerate(reader):
        total_records += 1

        try:
            year = int(row["Year"])
        except (ValueError, TypeError):
            continue
        if year != TARGET_YEAR:
            continue

        # Requirement 2: count incidents per attack type
        attack_counts[(row[COUNT_FIELD] or "Unknown").strip()] += 1

        # Handle invalid financial values (empty, non-numeric, negative)
        try:
            loss = float(row[LOSS_FIELD])
            if loss < 0:
                raise ValueError
        except (ValueError, TypeError):
            invalid_loss += 1
            loss = None
        else:
            total_loss += loss

        selected.append((order, loss, row))

incidents = len(selected)
valid_losses = [item for item in selected if item[1] is not None]

print("Total records in dataset:", total_records)
print("Incidents in target year:", incidents)
print("Invalid financial values skipped:", invalid_loss)

Total records in dataset: 3000
Incidents in target year: 285
Invalid financial values skipped: 0


In [13]:
#[CELL 4] ATTACK COUNTS AND FINANCIAL TOTALS
avg_loss = total_loss / len(valid_losses) if valid_losses else None

print("Count per attack type:")
for attack, count in attack_counts.most_common():
    print(f"  {attack}: {count}")

print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):",
      round(avg_loss, 2) if avg_loss is not None else "N/A")

Count per attack type:
  Phishing: 55
  DDoS: 53
  Man-in-the-Middle: 47
  SQL Injection: 47
  Ransomware: 42
  Malware: 41
Total financial loss (USD millions): 13947.26
Average financial loss (USD millions): 48.94


In [14]:
#[CELL 5] FIVE COSTLIEST INCIDENTS
top5 = sorted(valid_losses, key=lambda item: -item[1])[:5]

if not top5:
    print("No valid incidents found.")
for rank, (order, loss, row) in enumerate(top5, 1):
    print(f"{rank}. {row.get('Country', '?')} | {row[COUNT_FIELD]} | {loss} M")

1. Germany | Phishing | 99.98 M
2. USA | Ransomware | 99.88 M
3. Brazil | Malware | 99.29 M
4. USA | Ransomware | 99.19 M
5. Australia | Phishing | 98.72 M


In [15]:
#[CELL 6] EXPORT FILTERED CSV AND TOP-5 JSON
with open(filtered_path, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(row for _, _, row in selected)

top5_records = [
    {"rank": rank, "financial_loss_million_usd": loss, **row}
    for rank, (_, loss, row) in enumerate(top5, 1)
]
with open(json_path, "w", encoding="utf-8") as file:
    json.dump(top5_records, file, indent=4)

print("Filtered CSV created:", filtered_path)
print("JSON created:", json_path)

Filtered CSV created: output\Silverio_TUPM-26-1261_ex3_filtered.csv
JSON created: output\Silverio_TUPM-26-1261_ex3_top5.json


In [16]:
#[CELL 7] BACKUP AND VERIFY
shutil.copy2(json_path, backup_path)

backup_ok = (backup_path.exists()
             and backup_path.stat().st_size == json_path.stat().st_size)

print("Backup:", backup_path)
print("Backup exists:", backup_path.exists())
print("Backup verification:", "VERIFIED" if backup_ok else "FAILED")

Backup: backup\Silverio_TUPM-26-1261_ex3_top5.json
Backup exists: True
Backup verification: VERIFIED


In [17]:
#[CELL 8] GENERATE TEXT REPORT
with open(report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-----------------------------\n")
    report.write(f"Student: {LAST_NAME} ({STUDENT_ID})\n")
    report.write(f"Target year: {TARGET_YEAR}\n")
    report.write(f"Total records in dataset: {total_records}\n")
    report.write(f"Incidents in target year: {incidents}\n")
    report.write(f"Invalid financial values skipped: {invalid_loss}\n\n")

    report.write("Count per attack type:\n")
    for attack, count in attack_counts.most_common():
        report.write(f"  {attack}: {count}\n")

    report.write(f"\nTotal financial loss (USD millions): {total_loss:.2f}\n")
    report.write("Average financial loss (USD millions): "
                 + (f"{avg_loss:.2f}\n" if avg_loss is not None else "N/A\n"))

    report.write("\nFive costliest incidents:\n")
    if not top5:
        report.write("  No valid incidents found.\n")
    for rank, (_, loss, row) in enumerate(top5, 1):
        report.write(f"  {rank}. {row.get('Country', '?')} | {row[COUNT_FIELD]} | {loss:.2f}M\n")

    report.write("\nOutput files:\n")
    for path in (filtered_path, json_path, backup_path, report_path):
        report.write(f"  {path}\n")
    report.write(f"Backup verification: {'VERIFIED' if backup_ok else 'FAILED'}\n")

print("Report created:", report_path)

Report created: output\Silverio_TUPM-26-1261_ex3_report.txt


In [18]:
#[CELL 9] FINAL OUTPUT
print("=== EXERCISE 3: CYBERSECURITY INCIDENT REPORT ===")
print("Target year                  :", TARGET_YEAR)
print("Total records in dataset     :", total_records)
print("Incidents in target year     :", incidents)
print("Count per attack type        :", dict(attack_counts.most_common()))
print("Total financial loss (USD M) :", round(total_loss, 2))
print("Average financial loss (USD M):", round(avg_loss, 2) if avg_loss is not None else "N/A")
print("Five costliest incidents     :")
for rank, (_, loss, row) in enumerate(top5, 1):
    print(f"  {rank}. {row.get('Country', '?')} | {row[COUNT_FIELD]} | {loss} M")
print("Filtered CSV file            :", filtered_path)
print("JSON summary file            :", json_path)
print("Backup verification          :", "VERIFIED" if backup_ok else "FAILED")
print("Report file                  :", report_path)

=== EXERCISE 3: CYBERSECURITY INCIDENT REPORT ===
Target year                  : 2016
Total records in dataset     : 3000
Incidents in target year     : 285
Count per attack type        : {'Phishing': 55, 'DDoS': 53, 'Man-in-the-Middle': 47, 'SQL Injection': 47, 'Ransomware': 42, 'Malware': 41}
Total financial loss (USD M) : 13947.26
Average financial loss (USD M): 48.94
Five costliest incidents     :
  1. Germany | Phishing | 99.98 M
  2. USA | Ransomware | 99.88 M
  3. Brazil | Malware | 99.29 M
  4. USA | Ransomware | 99.19 M
  5. Australia | Phishing | 98.72 M
Filtered CSV file            : output\Silverio_TUPM-26-1261_ex3_filtered.csv
JSON summary file            : output\Silverio_TUPM-26-1261_ex3_top5.json
Backup verification          : VERIFIED
Report file                  : output\Silverio_TUPM-26-1261_ex3_report.txt
